In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# --- CONFIGURATION ---
num_units = 400
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

# 1. GENERATE UNITS TABLE
unit_types = {
    'Studio': {'weight': 0.15, 'sqft': (500, 600), 'rent': (1600, 1800)},
    '1BR': {'weight': 0.45, 'sqft': (700, 850), 'rent': (1900, 2200)},
    '2BR': {'weight': 0.30, 'sqft': (1000, 1200), 'rent': (2400, 2800)},
    '3BR': {'weight': 0.10, 'sqft': (1300, 1500), 'rent': (3000, 3400)}
}

units_data = []
unit_id = 1
for u_type, specs in unit_types.items():
    count = int(num_units * specs['weight'])
    for _ in range(count):
        sqft = np.random.randint(specs['sqft'][0], specs['sqft'][1])
        rent = np.random.randint(specs['rent'][0], specs['rent'][1])
        units_data.append({
            'Unit_ID': unit_id,
            'Unit_Number': f"{100 + (unit_id // 40) * 100 + (unit_id % 40)}",
            'Bedrooms': u_type,
            'Square_Feet': sqft,
            'Market_Rent': rent
        })
        unit_id += 1

df_units = pd.DataFrame(units_data)

# 2. GENERATE LEASES TABLE (Incorporating ~85% Retention)
leases_data = []
lease_id = 1
resident_id = 1000

for idx, row in df_units.iterrows():
    # Simulate historical and current year leases expiring in 2025
    num_historical_leases = np.random.randint(1, 3)
    current_lease_end = start_date + timedelta(days=np.random.randint(0, 360))

    # Current/Ending lease
    is_renewal_eligible = np.random.rand() < 0.95 # 95% are offered renewals
    accepted_renewal = np.random.rand() < 0.85 if is_renewal_eligible else False

    status = 'Renewed' if accepted_renewal else 'Vacated'

    leases_data.append({
        'Lease_ID': lease_id,
        'Unit_ID': row['Unit_ID'],
        'Resident_ID': resident_id,
        'Lease_Start_Date': (current_lease_end - timedelta(days=365)).strftime('%Y-%m-%d'),
        'Lease_End_Date': current_lease_end.strftime('%Y-%m-%d'),
        'Renewal_Offered': 'Y' if is_renewal_eligible else 'N',
        'Renewal_Accepted': 'Y' if accepted_renewal else 'N',
        'Lease_Status': status
    })

    # If they vacated, a new lease/resident fills it shortly after
    if status == 'Vacated' and current_lease_end < end_date:
        lease_id += 1
        resident_id += 1
        turn_days = np.random.randint(5, 20)
        new_start = current_lease_end + timedelta(days=turn_days)
        new_end = new_start + timedelta(days=365)

        leases_data.append({
            'Lease_ID': lease_id,
            'Unit_ID': row['Unit_ID'],
            'Resident_ID': resident_id,
            'Lease_Start_Date': new_start.strftime('%Y-%m-%d'),
            'Lease_End_Date': new_end.strftime('%Y-%m-%d'),
            'Renewal_Offered': 'N',
            'Renewal_Accepted': 'N',
            'Lease_Status': 'Active'
        })

    lease_id += 1
    resident_id += 1

df_leases = pd.DataFrame(leases_data)

# 3. GENERATE FINANCIAL TRANSACTIONS (With 2-8% Delinquency)
transactions = []
trans_id = 1

for idx, lease in df_leases.iterrows():
    start = datetime.strptime(lease['Lease_Start_Date'], '%Y-%m-%d')
    end = datetime.strptime(lease['Lease_End_Date'], '%Y-%m-%d')

    # Determine actual active months within our 2025 analysis window
    act_start = max(start, start_date)
    act_end = min(end, end_date)

    rent_amount = df_units[df_units['Unit_ID'] == lease['Unit_ID']]['Market_Rent'].values[0]

    # Loop through months
    current_month = act_start.replace(day=1)
    while current_month <= act_end:
        # Monthly Rent Charge
        charge_date = current_month.replace(day=1)

        # Monthly delinquency variance (2% to 8%)
        month_delinq_rate = np.random.uniform(0.02, 0.08)
        is_delinquent = np.random.rand() < month_delinq_rate

        transactions.append({
            'Transaction_ID': trans_id,
            'Lease_ID': lease['Lease_ID'],
            'Transaction_Date': charge_date.strftime('%Y-%m-%d'),
            'Category': 'Rent Charge',
            'Amount': rent_amount,
            'Status': 'Unpaid' if is_delinquent else 'Paid'
        })
        trans_id += 1

        # If paid, generate corresponding payment entry
        if not is_delinquent:
            pay_day = np.random.randint(1, 5) # Paid on time between 1st and 5th
            transactions.append({
                'Transaction_ID': trans_id,
                'Lease_ID': lease['Lease_ID'],
                'Transaction_Date': (charge_date + timedelta(days=pay_day-1)).strftime('%Y-%m-%d'),
                'Category': 'Payment',
                'Amount': -rent_amount,
                'Status': 'Paid'
            })
            trans_id += 1
        else:
            # Sometime late fee gets tacked on
            if np.random.rand() < 0.7:
                transactions.append({
                    'Transaction_ID': trans_id,
                    'Lease_ID': lease['Lease_ID'],
                    'Transaction_Date': (charge_date + timedelta(days=5)).strftime('%Y-%m-%d'),
                    'Category': 'Late Fee',
                    'Amount': 100,
                    'Status': 'Unpaid'
                })
                trans_id += 1

        # Move to next month
        if current_month.month == 12:
            break
        current_month = current_month.replace(month=current_month.month + 1)

df_transactions = pd.DataFrame(transactions)

# 4. GENERATE RESIDENT SURVEYS
surveys = []
survey_id = 1
touchpoints = ['Move-in', 'Maintenance Request', 'Renewal Notice Received']

for idx, lease in df_leases.iterrows():
    # Generate 1-2 random survey touches per lease
    for _ in range(np.random.randint(1, 3)):
        tp = np.random.choice(touchpoints)
        # Delinquent or non-renewing residents score lower on average
        if lease['Renewal_Accepted'] == 'N' and tp == 'Renewal Notice Received':
            score = np.random.choice([1, 2, 3, 4], p=[0.3, 0.4, 0.2, 0.1])
        else:
            score = np.random.choice([3, 4, 5], p=[0.1, 0.4, 0.5])

        surveys.append({
            'Survey_ID': survey_id,
            'Resident_ID': lease['Resident_ID'],
            'Touchpoint': tp,
            'Score': score,
            'Survey_Date': lease['Lease_Start_Date'] # Simplified timeline proxy
        })
        survey_id += 1

df_surveys = pd.DataFrame(surveys)

# --- SAVE TO CSV ---
df_units.to_csv('property_units.csv', index=False)
df_leases.to_csv('property_leases.csv', index=False)
df_transactions.to_csv('property_transactions.csv', index=False)
df_surveys.to_csv('property_surveys.csv', index=False)

print("Success! Generated 4 connected property dataset files.")

Success! Generated 4 connected property dataset files.
